# Lektion 05 - Agentisk RAG


## Opsætning

Denne notesbog demonstrerer Agentic RAG (Retrieval-Augmented Generation) mønsteret ved brug af Microsoft Agent Framework.

**Forudsætninger:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — dit Azure AI Search service-endpoint
- `AZURE_SEARCH_API_KEY` — din Azure AI Search API-nøgle
- Azure OpenAI deployment konfigureret via miljøvariabler
- Azure CLI autentificeret (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hvad er Agentic RAG?

Traditionel RAG følger en fast pipeline: hent dokumenter, og generer derefter et svar. **Agentic RAG** går videre ved at give agenten autonomi til at bestemme **hvornår** og **hvordan** information skal hentes.

Med Agentic RAG kan agenten:
- **Bestemme** om hentning er nødvendig, før et spørgsmål besvares
- **Vælge** hvilken datakilde eller værktøj der skal forespørges
- **Vurdere** hentede resultater og foretage opfølgende hentninger, hvis første forsøg er utilstrækkeligt
- **Kombinere** information fra flere hentningstrin til et sammenhængende svar

Dette gør agenten mere fleksibel og præcis sammenlignet med en statisk hent-og-generer pipeline.


## Oprettelse af et søgeværktøj

I Agentic RAG indkapsles eksterne datakilder som **værktøjer**, som agenten kan kalde efter behov. Dette giver agenten mulighed for at betragte opslag som blot en anden handling, den kan udføre, frem for et obligatorisk trin.

Nedenfor definerer vi en rejsevidensbase og eksponerer den som et værktøj, agenten kan kalde for at slå destinationsoplysninger op.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Bygning af RAG-agenten

Nu opretter vi en agent, der er instrueret i **altid at hente information, før den svarer**. Agenten bruger værktøjet `search_travel_knowledge` til at forankre sine svar i vidensbasen i stedet for at stole på sine egne træningsdata.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativ genfinding — Maker-Checker-mønsteret

En vigtig fordel ved Agentic RAG er **iterativ genfinding**. Agenten kan udføre flere runder med søgning for at verificere, forfine eller udvide sine oprindelige fund — svarende til en "maker-checker"-arbejdsgang:

1. **Maker-trin**: Agenten henter indledende information og udarbejder et svar.
2. **Checker-trin**: Agenten udfører yderligere genfindinger for at kontrollere detaljer eller udfylde huller.

Nedenfor bliver agenten stillet et spørgsmål, der kræver sammenligning af flere destinationer, hvilket får den til at søge flere gange.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Oversigt

I denne lektion lærte du at opbygge et **Agentic RAG**-system ved hjælp af Microsoft Agent Framework:

- **Agentic RAG** lader agenter selvstændigt beslutte, hvornår de skal hente information, hvilket gør hentningen dynamisk frem for fast.
- **Værktøjer som datakilder**: Eksterne vidensbaser (som Azure AI Search) indkapsles som værktøjer, som agenten kan kalde på.
- **Iterativ hentning**: maker-checker-mønstret gør det muligt for agenten at udføre flere runder med hentning — søge, verificere og forfine — før den producerer et endeligt svar.

I produktion vil du erstatte den midlertidige `TRAVEL_KNOWLEDGE_BASE` i hukommelsen med en rigtig Azure AI Search-indeks for at håndtere storstilet rejsedokumenthentning.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets modersmål betragtes som den autoritative kilde. For kritiske oplysninger anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for eventuelle misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
